# CNN Audio Classification with Backpropagation (Google Colab Version)

This notebook is designed for a **folder-based Google Drive dataset structure** and does **not** use zip extraction during runtime.

## Expected Google Drive structure

```text
MyDrive/
└── ML_project/
    ├── HS/
    ├── LS/
    ├── Mix/
    ├── HS.csv
    ├── LS.csv
    └── Mix.csv
```

## Project aim
This project classifies **heart sounds**, **lung sounds**, and **mixed sounds** by:
1. Reading metadata from CSV files
2. Loading `.wav` audio files from the `HS`, `LS`, and `Mix` folders
3. Converting audio into **Mel-spectrograms**
4. Training a **Convolutional Neural Network (CNN)**
5. Demonstrating **backpropagation** through TensorFlow training and gradient computation


In [ ]:
# Install required packages in Colab
!pip -q install librosa soundfile

In [ ]:
# Imports
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TensorFlow version:", tf.__version__)

## 1. Mount Google Drive and define the dataset path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the dataset location in Google Drive
DATA_DIR = Path('/content/drive/MyDrive/ML_project')

required_items = ['HS', 'LS', 'Mix', 'HS.csv', 'LS.csv', 'Mix.csv']

print("Checking dataset folder:", DATA_DIR)
for item in required_items:
    path = DATA_DIR / item
    print(f"{item:6s} -> {'FOUND' if path.exists() else 'MISSING'}")

assert DATA_DIR.exists(), f"Dataset folder not found: {DATA_DIR}" 

## 2. Inspect the folder contents

This cell helps verify that Colab can see the dataset files and folders correctly.

In [ ]:
print("Files and folders inside ML_project:")
for item in sorted(DATA_DIR.iterdir()):
    print("-", item.name)

## 3. Load metadata CSV files

In [ ]:
hs = pd.read_csv(DATA_DIR / 'HS.csv').copy()
ls = pd.read_csv(DATA_DIR / 'LS.csv').copy()
mix = pd.read_csv(DATA_DIR / 'Mix.csv').copy()

print("HS shape :", hs.shape)
print("LS shape :", ls.shape)
print("Mix shape:", mix.shape)

display(hs.head())
display(ls.head())
display(mix.head())

## 4. Build file paths and labels

The code below assumes the following metadata columns:
- `Heart Sound ID` in `HS.csv`
- `Lung Sound ID` in `LS.csv`
- `Mixed Sound ID` in `Mix.csv`

If your CSV headers are slightly different, update the column names in this cell only.

In [ ]:
hs['label'] = 'heart'
ls['label'] = 'lung'
mix['label'] = 'mixed'

hs['filepath'] = hs['Heart Sound ID'].astype(str).apply(lambda x: str(DATA_DIR / 'HS' / f'{x}.wav'))
ls['filepath'] = ls['Lung Sound ID'].astype(str).apply(lambda x: str(DATA_DIR / 'LS' / f'{x}.wav'))
mix['filepath'] = mix['Mixed Sound ID'].astype(str).apply(lambda x: str(DATA_DIR / 'Mix' / f'{x}.wav'))

audio_df = pd.concat([
    hs[['filepath', 'label']],
    ls[['filepath', 'label']],
    mix[['filepath', 'label']]
], ignore_index=True)

print("Combined dataset shape:", audio_df.shape)
display(audio_df.head())

## 5. Validate that audio files exist

This step is important for debugging. It confirms whether the `.wav` filenames match the IDs in the CSV files.

In [ ]:
audio_df['file_exists'] = audio_df['filepath'].apply(os.path.exists)

print(audio_df['file_exists'].value_counts())
display(audio_df[~audio_df['file_exists']].head())

assert audio_df['file_exists'].all(), (
    "Some audio files were not found. Check folder names, CSV IDs, or .wav filenames."
)

## 6. Convert audio files into Mel-spectrogram features

A Mel-spectrogram is used because it converts time-domain audio into an image-like time-frequency representation that is suitable for CNN input.

In [ ]:
# Audio feature settings
SAMPLE_RATE = 22050
DURATION = 6
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

def extract_mel_spectrogram(
    file_path: str,
    sr: int = SAMPLE_RATE,
    duration: int = DURATION,
    n_mels: int = N_MELS,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
) -> np.ndarray:
    """Load an audio file and convert it to a fixed-size Mel-spectrogram."""
    audio, sample_rate = librosa.load(file_path, sr=sr)

    target_length = sr * duration
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sample_rate,
        n_mels=n_mels,
        n_fft=n_fft,
        hop_length=hop_length,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalise to [0, 1]
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)

    return mel_db.astype(np.float32)

# Quick test on one file
sample_feature = extract_mel_spectrogram(audio_df['filepath'].iloc[0])
print("Mel-spectrogram shape:", sample_feature.shape)

## 7. Create feature matrix and encoded labels

In [ ]:
X = np.array([extract_mel_spectrogram(fp) for fp in audio_df['filepath']])
X = X[..., np.newaxis]  # Add channel dimension for CNN

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(audio_df['label'])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", label_encoder.classes_)

## 8. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape :", X_test.shape, y_test.shape)

## 9. Visualise one waveform and its Mel-spectrogram

In [ ]:
def show_waveform_and_mel(file_path: str, title_prefix: str) -> None:
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    target_length = sr * DURATION
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    plt.figure(figsize=(12, 4))
    librosa.display.waveshow(audio, sr=sr)
    plt.title(f'{title_prefix} - Waveform')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.show()

    plt.figure(figsize=(12, 4))
    librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'{title_prefix} - Mel-spectrogram')
    plt.xlabel('Time')
    plt.ylabel('Mel Frequency')
    plt.show()

show_waveform_and_mel(audio_df['filepath'].iloc[0], audio_df['label'].iloc[0])

## 10. Build the CNN model

The CNN learns spatial patterns from the Mel-spectrogram images. During training, backpropagation updates the trainable parameters to reduce classification loss.

In [ ]:
model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 11. Train the CNN

Backpropagation occurs inside `model.fit(...)`:
- forward pass produces predictions
- loss compares predictions to true labels
- gradients are computed by automatic differentiation
- optimiser updates the weights


In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

## 12. Explicit backpropagation demonstration with `GradientTape`

This cell provides direct evidence of gradient computation, which is the mathematical core of backpropagation.

In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam()

x_batch = tf.convert_to_tensor(X_train[:16])
y_batch = tf.convert_to_tensor(y_train[:16])

with tf.GradientTape() as tape:
    predictions = model(x_batch, training=True)      # Forward pass
    loss_value = loss_fn(y_batch, predictions)       # Loss calculation

gradients = tape.gradient(loss_value, model.trainable_variables)  # Backward pass

print("Loss on batch:", float(loss_value.numpy()))
print("Number of trainable tensors:", len(model.trainable_variables))
print("Number of gradient tensors :", len(gradients))

for i, grad in enumerate(gradients[:5]):
    print(f"Gradient {i+1} shape:", grad.shape if grad is not None else None)

## 13. Evaluate model performance

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## 14. Plot training curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

## 15. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False)
plt.title('Confusion Matrix')
plt.show()

## 16. Save the trained model

This makes the notebook more complete for assignment submission and reuse.

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/ML_project/cnn_audio_backprop_model.keras'
model.save(MODEL_PATH)
print("Model saved to:", MODEL_PATH)

## 17. Conclusion

This notebook implements a complete audio-classification pipeline using a folder-based Google Drive dataset. Audio signals are transformed into Mel-spectrograms, which are then classified by a CNN. Backpropagation is used during training to minimise classification error by propagating gradients backwards through the network and updating the trainable parameters.

For report writing, you can state that:
- the original data consisted of audio recordings
- the CNN learned from Mel-spectrogram representations of those recordings
- backpropagation was applied through TensorFlow's training process and explicitly demonstrated with `GradientTape`
